## Chroma
It is AI native open source vector db.(Apache 2.0)

In [12]:
##run this if you are using ubuntu system and facing sqllite version issue
# --- STEP 1: THE FIX (MUST BE FIRST) ---
import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# --- STEP 2: VERIFY ---
import sqlite3
print(f"Current SQLite version: {sqlite3.sqlite_version}")

Current SQLite version: 3.51.1


In [13]:
## Importing Modules
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma


In [14]:
#Load the text and split it and then convert into vectors
loader=TextLoader("../DataIngestion/speech.txt")
docs=loader.load()
docs

[Document(metadata={'source': '../DataIngestion/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of righ

In [15]:
#Splitting
documents_split=CharacterTextSplitter(chunk_size=1000,chunk_overlap=50).split_documents(docs)
documents_split

[Document(metadata={'source': '../DataIngestion/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': '../DataIngestion/speech.txt'}, page_content='…\n\nIt will be al

In [16]:
#vector Conversion
Embedding=OllamaEmbeddings(model="all-minilm:latest")

In [17]:
#converting into vectors and storing in the Chroma DB
chroma_db=Chroma.from_documents(documents_split,Embedding,persist_directory="./chroma_db")
chroma_db

In [18]:
##query the vector
query="How does Wilson distinguish between the German government and the German people? Why was this distinction strategically important in 1917?"
answer=chroma_db.similarity_search(query)
answer

[Document(id='88ddf7c4-8016-46b7-836c-b7c6a22c748b', metadata={'source': '../DataIngestion/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='cd4e09c9-59cc-4956-825b-e7b0055d3a70', metadata={'source': '../DataIngestion/speech.txt'}, page_content='We have borne with their present government through all these bitter months because o

In [19]:
##loading from the disk
chroma_db_loaded=Chroma(persist_directory="./chroma_db",embedding_function=Embedding)
chroma_db_loaded

In [20]:
## Retriever Class
# the vector stores can be used as retriever classes which are used by RAG pipelines
retriever=chroma_db_loaded.as_retriever()
response=retriever.invoke("How many months of trail?")
response

[Document(id='d85112b5-0c4b-4a7c-8e43-0ccd48edc344', metadata={'source': '../DataIngestion/speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
 Document(id='d1990e10-007e-4cd1-af93-caa4b6d1d794', metadata={'source': '../DataIngestion/speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, a

In [21]:
# we can retrieve the output based on the similarity search score
# we use the distance for the score calculation. less dist means high matching
query="Do we still have the opportunity to prove taht friendship?"
answer=chroma_db_loaded.similarity_search_with_score(query)
answer

[(Document(id='cd4e09c9-59cc-4956-825b-e7b0055d3a70', metadata={'source': '../DataIngestion/speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a patience and forbearance which would otherwise have been impossible. We shall, happily, still have an opportunity to prove that friendship in our daily attitude and actions toward the millions of men and women of German birth and native sympathy who live among us and share our life, and we shall be proud to prove it toward all who are in fact loyal to their neighbors and to the government in the hour of test. They are, most of them, as true and loyal Americans as if they had never known any other fealty or allegiance. They will be prompt to stand with us in rebuking and restraining the few who may be of a different mind and purpose. If there should be disloyalty, it will be dealt with with a firm hand of stern repression; but, if it lifts its head at all

In [22]:
# we can search based on the vectors
embedding_vector=Embedding.embed_query(query)
embedding_vector

[-0.08885857,
 0.022331497,
 0.08200906,
 -0.03717838,
 0.004003343,
 -0.018891867,
 0.034863062,
 -0.076765746,
 0.05133195,
 -0.0346508,
 -0.00036389063,
 0.010775869,
 0.018520921,
 0.0870759,
 0.017639583,
 -0.0350127,
 -0.06068629,
 -0.03379682,
 -0.041745793,
 -0.011827114,
 -0.08359431,
 -0.059207644,
 0.0025727605,
 -0.06324252,
 -0.033499002,
 -0.058009252,
 0.06659328,
 0.0029703947,
 0.025870984,
 0.005644082,
 0.07623715,
 0.027720563,
 -0.01053019,
 -0.03340866,
 0.020251542,
 0.08549813,
 0.08039614,
 -0.04076426,
 0.050243698,
 -0.028817859,
 -0.016996382,
 -0.0921592,
 0.007625184,
 -0.039900906,
 -0.097733274,
 0.038800213,
 0.034632895,
 0.022226788,
 -0.12249016,
 -0.020693867,
 -0.058238912,
 0.030233433,
 -0.017622652,
 0.006677573,
 0.13934062,
 0.03236606,
 0.05253012,
 0.02147636,
 -0.053398564,
 -0.04642281,
 -0.058014948,
 -0.013474987,
 0.004072562,
 0.034910206,
 0.025102418,
 0.0049810903,
 -0.035474427,
 0.055603128,
 -0.01670176,
 0.020897087,
 0.01566172

In [23]:
#result with score ,search with vectors
result_score=chroma_db_loaded.similarity_search_by_vector(embedding_vector)
result_score

[Document(id='cd4e09c9-59cc-4956-825b-e7b0055d3a70', metadata={'source': '../DataIngestion/speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a patience and forbearance which would otherwise have been impossible. We shall, happily, still have an opportunity to prove that friendship in our daily attitude and actions toward the millions of men and women of German birth and native sympathy who live among us and share our life, and we shall be proud to prove it toward all who are in fact loyal to their neighbors and to the government in the hour of test. They are, most of them, as true and loyal Americans as if they had never known any other fealty or allegiance. They will be prompt to stand with us in rebuking and restraining the few who may be of a different mind and purpose. If there should be disloyalty, it will be dealt with with a firm hand of stern repression; but, if it lifts its head at all,